# Giai đoạn 4: Advanced NLP - Aspect-Based Sentiment Analysis (ABSA)

Sử dụng PyTorch và HuggingFace Transformers để huấn luyện một mô hình Multi-label ABSA.

## Dữ liệu nhãn (pseudo-label)
Notebook này sẽ load file nhãn được tạo từ notebook `03b_ABSA_AutoLabeling.ipynb`:
- `Notebook_Report/absa/labeled_absa_auto.jsonl`

Lưu ý: Phần code phía dưới train/val trên pseudo-label (từ `03b_ABSA_AutoLabeling.ipynb`) và in `val_microF1/val_macroF1`.


In [1]:
import random

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer

ASPECTS = ["script", "acting", "visuals", "music", "pacing", "direction", "overall"]
SENTIMENTS = ["negative", "neutral", "positive"]
NUM_LABELS = len(ASPECTS) * len(SENTIMENTS)


def get_label_idx(aspect: str, sentiment: str) -> int:
    if aspect in ASPECTS and sentiment in SENTIMENTS:
        return ASPECTS.index(aspect) * len(SENTIMENTS) + SENTIMENTS.index(sentiment)
    return -1


class DummyAbsaDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=64):
        self.samples = []
        for item in data:
            vec = [0.0] * NUM_LABELS
            for l in item["labels"]:
                idx = get_label_idx(l["aspect"], l["sentiment"])
                if idx >= 0:
                    vec[idx] = 1.0
            self.samples.append((item["text"], vec))
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        text, vec = self.samples[i]
        enc = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(vec, dtype=torch.float32),
        }


class AbsaClassifier(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.head = nn.Linear(self.backbone.config.hidden_size, NUM_LABELS)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.head(cls)


def predict(model: nn.Module, loader: DataLoader, threshold: float = 0.5, device: str = "cpu"):
    model.eval()
    ys = []
    yh = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()
            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= threshold).astype(int)
            ys.append(labels)
            yh.append(preds)
    y_true = np.vstack(ys) if ys else np.zeros((0, NUM_LABELS))
    y_pred = np.vstack(yh) if yh else np.zeros((0, NUM_LABELS))
    return y_true, y_pred


print("Bo qua dummy-data ABSA mô phỏng vì dummy_data có thể không tồn tại.")
print("Cell train thật sẽ dùng pseudo-label ở cell kế tiếp (labeled_absa_auto.jsonl).")


Bo qua dummy-data ABSA mô phỏng vì dummy_data có thể không tồn tại.
Cell train thật sẽ dùng pseudo-label ở cell kế tiếp (labeled_absa_auto.jsonl).


In [2]:
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer

# -----------------------------
# Schema label (đóng gói trong Notebook_Report)
# -----------------------------
ASPECTS = ["script", "acting", "visuals", "music", "pacing", "direction", "overall"]
SENTIMENTS = ["negative", "neutral", "positive"]
NUM_LABELS = len(ASPECTS) * len(SENTIMENTS)


def get_label_index(aspect: str, sentiment: str) -> int:
    if aspect not in ASPECTS or sentiment not in SENTIMENTS:
        raise ValueError(f"Unknown aspect={aspect!r} or sentiment={sentiment!r}")
    return ASPECTS.index(aspect) * len(SENTIMENTS) + SENTIMENTS.index(sentiment)


# -----------------------------
# Load pseudo-labels (output từ Notebook 03b)
# -----------------------------
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "Notebook_Report":
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "Notebook_Report" / "absa" / "labeled_absa_auto.jsonl").exists():
            NOTEBOOK_DIR = p / "Notebook_Report"
            break

LABELED_JSONL = NOTEBOOK_DIR / "absa" / "labeled_absa_auto.jsonl"
if not LABELED_JSONL.exists():
    raise FileNotFoundError(
        "Không thấy file nhãn ABSA: "
        f"{LABELED_JSONL} (cwd={Path.cwd()}). "
        "Hãy chạy notebook `03b_ABSA_AutoLabeling.ipynb` để tạo `Notebook_Report/absa/labeled_absa_auto.jsonl` trước."
    )

print("Notebook dir:", NOTEBOOK_DIR)
print("Labeled JSONL:", LABELED_JSONL)


# -----------------------------
# Cấu hình train (full data)
# -----------------------------
SEED = 42
MODEL_NAME = "prajjwal1/bert-tiny"

# FULL DATA: để None để đọc hết; đặt số (vd 2000) nếu muốn debug nhanh
LIMIT_TOTAL = None

VAL_RATIO = 0.2  # 80/20
EPOCHS = 5
BATCH_SIZE = 8
LR = 5e-4
MAX_LENGTH = 96
THRESHOLD = 0.5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def load_jsonl(path: Path, limit: int | None = None):
    out = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)
            text = ex.get("text", "")
            labels = ex.get("labels", [])
            if text and isinstance(labels, list):
                out.append({"text": text, "labels": labels})
    return out


examples = load_jsonl(LABELED_JSONL, limit=LIMIT_TOTAL)
if not examples:
    raise RuntimeError(f"File nhãn rỗng/không hợp lệ: {LABELED_JSONL}")

rng = random.Random(SEED)
rng.shuffle(examples)

val_size = max(1, int(round(VAL_RATIO * len(examples))))
val_examples = examples[:val_size]
train_examples = examples[val_size:]

print(f"Loaded: total={len(examples)} | train={len(train_examples)} | val={len(val_examples)}")


def vectorize_labels(label_list: list[dict]) -> np.ndarray:
    vec = np.zeros(NUM_LABELS, dtype=np.float32)
    for l in label_list:
        try:
            idx = get_label_index(l["aspect"], l["sentiment"])
            vec[idx] = 1.0
        except Exception:
            continue
    return vec


class AbsaDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=96):
        self.samples = []
        for item in data:
            self.samples.append((item["text"], vectorize_labels(item.get("labels", []))))
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        text, vec = self.samples[i]
        enc = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(vec, dtype=torch.float32),
        }


class AbsaClassifier(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.head = nn.Linear(self.backbone.config.hidden_size, NUM_LABELS)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.head(cls)


def predict(model: nn.Module, loader: DataLoader, threshold: float, device: str):
    model.eval()
    ys = []
    yh = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()

            logits = model(input_ids, attention_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= threshold).astype(int)

            ys.append(labels)
            yh.append(preds)

    return np.vstack(ys), np.vstack(yh)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds = AbsaDataset(train_examples, tokenizer, max_length=MAX_LENGTH)
val_ds = AbsaDataset(val_examples, tokenizer, max_length=MAX_LENGTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

model = AbsaClassifier(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss()

print("--- Train Loop ABSA (FULL pseudo-labels + validation micro/macro F1) ---")
for epoch in range(EPOCHS):
    model.train()
    losses = []

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.item()))

    y_true, y_pred = predict(model, val_loader, threshold=THRESHOLD, device=device)
    micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    print(f"Epoch {epoch + 1}/{EPOCHS} | loss={np.mean(losses):.4f} | val_microF1={micro:.3f} | val_macroF1={macro:.3f}")

print("Xong ABSA train (full pseudo-labels)!")

# -----------------------------
# Evaluate + export results cho Notebook 05
# -----------------------------
from collections import Counter

# 1) Predict trên tập val
y_true, y_pred = predict(model, val_loader, threshold=THRESHOLD, device=device)

micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

# 2) Per-label F1 (để minh hoạ chi tiết hơn)
per_label = []
for a in ASPECTS:
    for s in SENTIMENTS:
        idx = get_label_index(a, s)
        f1 = f1_score(y_true[:, idx], y_pred[:, idx], average="binary", zero_division=0)
        per_label.append({"aspect": a, "sentiment": s, "f1": float(f1)})

# 3) Confusion matrix 3x3 cho sentiment của aspect 'overall'
# (vì pseudo-label hiện tại gán 1 sentiment chung cho toàn câu, nên overall là đại diện hợp lý)
idx_overall = [get_label_index("overall", s) for s in SENTIMENTS]

true_cls = np.argmax(y_true[:, idx_overall], axis=1)
pred_cls = np.argmax(y_pred[:, idx_overall], axis=1)

cm3 = np.zeros((3, 3), dtype=int)
for t, p in zip(true_cls, pred_cls):
    cm3[int(t), int(p)] += 1

# 4) Lưu vài ví dụ dự đoán
label_names = [(a, s) for a in ASPECTS for s in SENTIMENTS]

sample_rows = []
for i in range(min(8, len(val_examples))):
    text = val_examples[i]["text"]
    tlabs = [f"{a}:{s}" for (a, s), v in zip(label_names, y_true[i]) if v >= 0.5]
    plabs = [f"{a}:{s}" for (a, s), v in zip(label_names, y_pred[i]) if v >= 0.5]
    sample_rows.append({"text": text[:300], "true_labels": tlabs[:12], "pred_labels": plabs[:12]})

# 5) Export JSON để Notebook 05 load
out_dir = NOTEBOOK_DIR / "absa"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "absa_eval.json"

payload = {
    "model": MODEL_NAME,
    "n_total": int(len(examples)),
    "n_train": int(len(train_examples)),
    "n_val": int(len(val_examples)),
    "micro_f1": float(micro),
    "macro_f1": float(macro),
    "overall_sentiment_cm": cm3.tolist(),
    "sentiments": SENTIMENTS,
    "per_label_f1": per_label,
    "samples": sample_rows,
}

with out_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"Đã lưu ABSA eval -> {out_path}")
print(f"microF1={micro:.3f} | macroF1={macro:.3f}")

# -----------------------------
# Save model weights (checkpoint) + tokenizer + metadata
# -----------------------------
artifact_dir = out_dir / "artifacts" / "absa_bert_tiny_latest"
artifact_dir.mkdir(parents=True, exist_ok=True)

# Lưu trọng số PyTorch
ckpt_path = artifact_dir / "model.pt"
torch.save(
    {
        "state_dict": model.state_dict(),
        "model_name": MODEL_NAME,
        "num_labels": int(NUM_LABELS),
        "aspects": ASPECTS,
        "sentiments": SENTIMENTS,
        "max_length": int(MAX_LENGTH),
        "threshold": float(THRESHOLD),
        "val_micro_f1": float(micro),
        "val_macro_f1": float(macro),
    },
    ckpt_path,
)

# Lưu tokenizer (để inference/reload thuận tiện)
try:
    tok_dir = artifact_dir / "tokenizer"
    tokenizer.save_pretrained(tok_dir)
except Exception as e:
    print("Cảnh báo: không lưu được tokenizer:", repr(e))

# Lưu metadata riêng (đọc dễ)
meta_path = artifact_dir / "metadata.json"
with meta_path.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "model_name": MODEL_NAME,
            "schema": {"aspects": ASPECTS, "sentiments": SENTIMENTS},
            "train": {
                "seed": int(SEED),
                "val_ratio": float(VAL_RATIO),
                "epochs": int(EPOCHS),
                "batch_size": int(BATCH_SIZE),
                "lr": float(LR),
                "max_length": int(MAX_LENGTH),
                "threshold": float(THRESHOLD),
            },
            "data": {"n_total": int(len(examples)), "n_train": int(len(train_examples)), "n_val": int(len(val_examples))},
            "val": {"micro_f1": float(micro), "macro_f1": float(macro)},
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Đã lưu checkpoint -> {ckpt_path}")
print(f"Đã lưu metadata   -> {meta_path}")


Notebook dir: /Users/kotori/CineSen/Notebook_Report
Labeled JSONL: /Users/kotori/CineSen/Notebook_Report/absa/labeled_absa_auto.jsonl
Loaded: total=9472 | train=7578 | val=1894


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertModel LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- Train Loop ABSA (FULL pseudo-labels + validation micro/macro F1) ---
Epoch 1/5 | loss=0.3379 | val_microF1=0.461 | val_macroF1=0.185
Epoch 2/5 | loss=0.2881 | val_microF1=0.523 | val_macroF1=0.228
Epoch 3/5 | loss=0.2365 | val_microF1=0.550 | val_macroF1=0.294
Epoch 4/5 | loss=0.1804 | val_microF1=0.563 | val_macroF1=0.369
Epoch 5/5 | loss=0.1360 | val_microF1=0.577 | val_macroF1=0.383
Xong ABSA train (full pseudo-labels)!
Đã lưu ABSA eval -> /Users/kotori/CineSen/Notebook_Report/absa/absa_eval.json
microF1=0.577 | macroF1=0.383
Đã lưu checkpoint -> /Users/kotori/CineSen/Notebook_Report/absa/artifacts/absa_bert_tiny_latest/model.pt
Đã lưu metadata   -> /Users/kotori/CineSen/Notebook_Report/absa/artifacts/absa_bert_tiny_latest/metadata.json


In [3]:
# -----------------------------
# Build movie-level ABSA profiles (for ranking refinement)
# -----------------------------
# Ý tưởng: aggregate score theo từng (aspect, sentiment) ở mức phim,
# để sau này (Notebook 06) có thể cộng "ABSA bonus" khi query có từ khoá cảm xúc.

from collections import defaultdict

import pandas as pd

ABSA_CLEAN_CSV = NOTEBOOK_DIR / "absa_clean_reviews.csv"
if not ABSA_CLEAN_CSV.exists():
    raise FileNotFoundError(
        f"Không thấy {ABSA_CLEAN_CSV}. Hãy chạy notebook 02 để export absa_clean_reviews.csv trước."
    )

df_absa = pd.read_csv(ABSA_CLEAN_CSV).fillna("")
if "tmdb_id" not in df_absa.columns or "cleaned_content" not in df_absa.columns:
    raise ValueError("absa_clean_reviews.csv cần có cột tmdb_id và cleaned_content")

# Giảm tải inference: lấy tối đa N reviews mỗi phim
MAX_REVIEWS_PER_MOVIE = 5
rows = (
    df_absa[df_absa["cleaned_content"].astype(str).str.len() >= 15]
    .groupby("tmdb_id")
    .head(MAX_REVIEWS_PER_MOVIE)
    .reset_index(drop=True)
)

print("ABSA clean rows (sampled per movie):", len(rows))

# Prepare label names
label_names = [(a, s) for a in ASPECTS for s in SENTIMENTS]

# Accumulators: tmdb_id -> idx -> (sum_prob, count)
acc_sum = defaultdict(lambda: np.zeros(NUM_LABELS, dtype=np.float32))
acc_cnt = defaultdict(int)

BATCH = 32
model.eval()

texts = rows["cleaned_content"].astype(str).tolist()
movie_ids = rows["tmdb_id"].astype(int).tolist() if str(rows["tmdb_id"].dtype).startswith("int") else rows["tmdb_id"].astype(str).tolist()

with torch.no_grad():
    for start in range(0, len(texts), BATCH):
        batch_text = texts[start : start + BATCH]
        batch_mid = movie_ids[start : start + BATCH]
        enc = tokenizer(
            batch_text,
            max_length=MAX_LENGTH,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)
        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits).detach().cpu().numpy()  # (B, NUM_LABELS)

        for mid, p in zip(batch_mid, probs):
            acc_sum[str(mid)] += p.astype(np.float32)
            acc_cnt[str(mid)] += 1

# Build average profile
movie_profiles = {}
for mid, vec_sum in acc_sum.items():
    cnt = acc_cnt[mid]
    if cnt <= 0:
        continue
    avg = (vec_sum / float(cnt)).tolist()

    # reshape: aspect -> sentiment -> score
    profile = {a: {s: 0.0 for s in SENTIMENTS} for a in ASPECTS}
    for (a, s), score in zip(label_names, avg):
        profile[a][s] = float(round(score, 4))

    movie_profiles[mid] = {
        "n_reviews": int(cnt),
        "scores": profile,
    }

out_profiles_path = NOTEBOOK_DIR / "absa" / "absa_movie_profiles.json"
with out_profiles_path.open("w", encoding="utf-8") as f:
    json.dump(movie_profiles, f, ensure_ascii=False, indent=2)

print(f"Đã lưu movie-level ABSA profiles -> {out_profiles_path} (movies={len(movie_profiles)})")

# -----------------------------
# Query -> Aspect intent mapping (heuristic)
# -----------------------------
# Dùng cho bonus scoring ở Notebook 06.

ASPECT_KEYWORDS = {
    "overall": ["hay", "xuất sắc", "good", "great", "amazing", "excellent"],
    "script": ["kịch bản", "plot", "story", "twist", "mind-bending"],
    "visuals": ["kỹ xảo", "cgi", "visual", "cinematography", "beautiful"],
    "acting": ["diễn xuất", "acting", "actor", "performance"],
    "pacing": ["nhịp", "pace", "slow", "boring", "drag"],
    "music": ["nhạc", "music", "soundtrack", "score"],
    "direction": ["đạo diễn", "direction", "director"],
}

def infer_query_aspects(query: str) -> list[str]:
    q = str(query).lower()
    hits = []
    for aspect, kws in ASPECT_KEYWORDS.items():
        if any(kw in q for kw in kws):
            hits.append(aspect)
    return hits

print("Example query aspects:", infer_query_aspects("great visuals but pacing too slow"))


ABSA clean rows (sampled per movie): 7659
Đã lưu movie-level ABSA profiles -> /Users/kotori/CineSen/Notebook_Report/absa/absa_movie_profiles.json (movies=2426)
Example query aspects: ['overall', 'visuals', 'pacing']
